In [1]:

%pip install -q pandas numpy pillow opencv-python-headless pytesseract pymupdf scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
import os, re, io, json, sqlite3, hashlib, unicodedata
import pandas as pd
import numpy as np
from PIL import Image, ImageOps
import cv2
import fitz
import pytesseract
from pytesseract import Output
from IPython.display import display

BASE = Path.cwd()

# Si Jupyter está ejecutándose dentro de entrenamiento,
# subir un nivel hasta la raíz del proyecto.
if BASE.name.lower() == "entrenamiento":
    BASE = BASE.parent

ENTRENAMIENTO = BASE / "entrenamiento"
MODELO_RUTA = BASE / "modelo_documentos.joblib"

ENTRADA = BASE / "documentos"
SALIDA = BASE / "resultados"

ENTRADA.mkdir(parents=True, exist_ok=True)
SALIDA.mkdir(parents=True, exist_ok=True)

print("Carpeta raíz:", BASE)
print("Carpeta de entrenamiento:", ENTRENAMIENTO)
print("¿Existe entrenamiento?:", ENTRENAMIENTO.exists())
print("Carpeta de entrada:", ENTRADA)
print("Carpeta de resultados:", SALIDA)

EXTENSIONES = {
    ".pdf", ".jpg", ".jpeg", ".png",
    ".tif", ".tiff", ".bmp", ".webp"
}

RUTA_TESSERACT = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    if os.name == "nt" else None
)

if RUTA_TESSERACT and Path(RUTA_TESSERACT).exists():
    pytesseract.pytesseract.tesseract_cmd = RUTA_TESSERACT

try:
    VERSION_OCR = str(pytesseract.get_tesseract_version())
    IDIOMAS = set(pytesseract.get_languages(config=""))
    OCR_OK = True
except Exception:
    VERSION_OCR = "no disponible"
    IDIOMAS = set()
    OCR_OK = False

if {"spa", "eng"}.issubset(IDIOMAS):
    IDIOMA_OCR = "spa+eng"
elif "spa" in IDIOMAS:
    IDIOMA_OCR = "spa"
elif "eng" in IDIOMAS:
    IDIOMA_OCR = "eng"
else:
    IDIOMA_OCR = ""

MAX_PAGINAS_PDF = 200
MIN_CARACTERES_TEXTO = 35
DPI_PDF = 220

print("Carpeta de entrada:", ENTRADA.resolve())
print("Carpeta de resultados:", SALIDA.resolve())
print("OCR:", VERSION_OCR)
print("Idioma:", IDIOMA_OCR or "ninguno")


Carpeta raíz: c:\Users\mcgre\Desktop\organizador_notebook_web
Carpeta de entrenamiento: c:\Users\mcgre\Desktop\organizador_notebook_web\entrenamiento
¿Existe entrenamiento?: True
Carpeta de entrada: c:\Users\mcgre\Desktop\organizador_notebook_web\documentos
Carpeta de resultados: c:\Users\mcgre\Desktop\organizador_notebook_web\resultados
Carpeta de entrada: C:\Users\mcgre\Desktop\organizador_notebook_web\documentos
Carpeta de resultados: C:\Users\mcgre\Desktop\organizador_notebook_web\resultados
OCR: 5.5.3.20260724
Idioma: eng


In [3]:

def limpiar_texto(texto):
    texto = (texto or "").replace("\x00", " ")
    return "\n".join(
        re.sub(r"[ \t]+", " ", linea).strip()
        for linea in texto.splitlines()
    ).strip()


def normalizar(texto):
    return "".join(
        c for c in unicodedata.normalize(
            "NFKD", (texto or "").lower()
        )
        if not unicodedata.combining(c)
    )


def estrategias_imagen(imagen):
    rgb = np.array(imagen.convert("RGB"))
    gris = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    alto, ancho = gris.shape

    if max(alto, ancho) < 1800:
        factor = min(2.0, 1800 / max(alto, ancho))
        gris = cv2.resize(
            gris, None,
            fx=factor, fy=factor,
            interpolation=cv2.INTER_CUBIC
        )

    gris = cv2.fastNlMeansDenoising(gris, h=8)

    adaptive = cv2.adaptiveThreshold(
        gris, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 35, 11
    )

    return [gris, adaptive]


def ocr_imagen(imagen):
    if not OCR_OK or not IDIOMA_OCR:
        return "", 0.0, "ocr_no_disponible"

    candidatos = []

    for n_imagen, img in enumerate(estrategias_imagen(imagen)):
        for psm in (3, 6):
            try:
                config = f"--oem 3 --psm {psm}"

                datos = pytesseract.image_to_data(
                    img,
                    lang=IDIOMA_OCR,
                    config=config,
                    output_type=Output.DICT
                )

                palabras = []
                valores = []

                for palabra, conf in zip(
                    datos["text"], datos["conf"]
                ):
                    if palabra and palabra.strip():
                        palabras.append(palabra.strip())
                        try:
                            if float(conf) >= 0:
                                valores.append(float(conf))
                        except (ValueError, TypeError):
                            pass

                texto = limpiar_texto(
                    pytesseract.image_to_string(
                        img, lang=IDIOMA_OCR, config=config
                    )
                )

                confianza = (
                    round(float(np.mean(valores)), 1)
                    if valores else 0.0
                )

                puntuacion = (
                    confianza + min(len(palabras), 100) * 0.03
                )

                candidatos.append((
                    puntuacion, texto, confianza,
                    f"ocr_{n_imagen}_psm{psm}"
                ))

            except Exception:
                continue

    if not candidatos:
        return "", 0.0, "error_ocr"

    _, texto, confianza, metodo = max(
        candidatos,
        key=lambda item: (item[0], len(item[1]))
    )

    return texto, confianza, metodo


def leer_documento(ruta):
    ruta = Path(ruta)
    paginas = []
    avisos = []

    if ruta.suffix.lower() == ".pdf":
        with fitz.open(ruta) as doc:
            if doc.needs_pass:
                raise ValueError(
                    "PDF protegido con contraseña"
                )

            if len(doc) > MAX_PAGINAS_PDF:
                avisos.append(
                    f"Solo se procesaron {MAX_PAGINAS_PDF} "
                    f"de {len(doc)} páginas"
                )

            for i in range(min(len(doc), MAX_PAGINAS_PDF)):
                pagina = doc[i]

                texto = limpiar_texto(
                    pagina.get_text("text", sort=True)
                )

                metodo = "texto_pdf"
                confianza = None

                cantidad = len(re.sub(r"\s", "", texto))

                if cantidad < MIN_CARACTERES_TEXTO:
                    try:
                        pix = pagina.get_pixmap(
                            matrix=fitz.Matrix(
                                DPI_PDF / 72,
                                DPI_PDF / 72
                            ),
                            alpha=False
                        )

                        imagen = Image.open(
                            io.BytesIO(pix.tobytes("png"))
                        )

                        texto_ocr, confianza, metodo_ocr = (
                            ocr_imagen(imagen)
                        )

                        if len(texto_ocr) > len(texto):
                            texto = texto_ocr
                            metodo = metodo_ocr

                    except Exception as e:
                        avisos.append(
                            f"Página {i+1}: error OCR ({e})"
                        )

                paginas.append({
                    "pagina": i + 1,
                    "texto": texto,
                    "metodo": metodo,
                    "confianza_ocr": confianza
                })

    else:
        with Image.open(ruta) as im:
            n_frames = min(
                getattr(im, "n_frames", 1),
                MAX_PAGINAS_PDF
            )

            for i in range(n_frames):
                im.seek(i)

                imagen = ImageOps.exif_transpose(
                    im.copy().convert("RGB")
                )

                texto, confianza, metodo = ocr_imagen(
                    imagen
                )

                paginas.append({
                    "pagina": i + 1,
                    "texto": texto,
                    "metodo": metodo,
                    "confianza_ocr": confianza
                })

    return paginas, avisos


In [4]:

CATEGORIAS = {
    "Factura": {
        "factura": 4,
        "subtotal": 2,
        "iva": 1,
        "factura de venta": 4
    },
    "Orden de compra": {
        "orden de compra": 6,
        "orden de pedido": 3,
        "precio unitario": 2
    },
    "Recibo": {
        "recibo": 4,
        "recibi": 2,
        "recibo de caja": 5
    },
    "Contrato": {
        "contrato": 5,
        "clausula": 2,
        "contratante": 2,
        "contratista": 2
    },
    "Formulario": {
        "formulario": 5,
        "fecha de nacimiento": 2,
        "apellidos": 1
    },
    "Comprobante": {
        "comprobante": 4,
        "transaccion": 2,
        "transferencia": 1,
        "referencia": 1
    }
}


def clasificar(texto):
    t = normalizar(texto)

    puntajes = {
        categoria: sum(
            peso
            for termino, peso in indicadores.items()
            if termino in t
        )
        for categoria, indicadores in CATEGORIAS.items()
    }

    orden = sorted(
        puntajes,
        key=puntajes.get,
        reverse=True
    )

    primero = puntajes[orden[0]]
    segundo = puntajes[orden[1]]

    if primero < 3 or primero == segundo:
        return "Revisión necesaria", puntajes

    return orden[0], puntajes


def buscar(texto, patrones):
    for patron in patrones:
        coincidencia = re.search(
            patron, texto, re.I | re.M
        )

        if coincidencia:
            valor = coincidencia.group(1).strip()
            valor = valor.strip(" -:#")

            if valor:
                return valor[:200]

    return None


def extraer_campos(texto):
    numero = buscar(texto, [
        r"^\s*(?:factura|recibo|comprobante|orden(?: de compra)?)\s*(?:de venta)?\s*(?:n[°ºo.]?\s*)?[:#-]?\s*([A-Z0-9][A-Z0-9\-/]{2,})\s*$",
        r"^\s*(?:n[úu]mero de (?:factura|documento)|folio|factura n[°ºo.]?)\s*[:#-]\s*([A-Z0-9\-/]{3,})"
    ])

    fecha = buscar(texto, [
        r"\b(\d{4}[-/]\d{1,2}[-/]\d{1,2})\b",
        r"\b(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})\b"
    ])

    nit = buscar(texto, [
        r"\bNIT\s*[:#-]?\s*(\d[\d.\- ]{4,18}\d)\b"
    ])

    totales = re.findall(
        r"^\s*(?:TOTAL(?: A PAGAR)?|VALOR TOTAL|IMPORTE TOTAL)\s*[:$ ]+\s*\$?\s*([\d][\d., ]{0,20})\s*$",
        texto,
        re.I | re.M
    )

    total = totales[-1].strip() if totales else None

    cliente = buscar(texto, [
        r"^\s*(?:cliente|raz[oó]n social del cliente)\s*[:\-]\s*(.{3,100})$"
    ])

    proveedor = buscar(texto, [
        r"^\s*(?:proveedor|emisor)\s*[:\-]\s*(.{3,100})$"
    ])

    return {
        "numero_documento": numero,
        "fecha": fecha,
        "nit": nit,
        "total": total,
        "cliente": cliente,
        "proveedor": proveedor
    }


texto_demo = """
FACTURA DE VENTA
NIT: 900123456-7
Cliente: Empresa Ejemplo
Fecha: 10/09/2026
Subtotal: 100000
IVA: 19000
TOTAL: 119000
"""

print("Categoría:", clasificar(texto_demo)[0])
print("Campos:", extraer_campos(texto_demo))


Categoría: Factura
Campos: {'numero_documento': None, 'fecha': '10/09/2026', 'nit': '900123456-7', 'total': '119000', 'cliente': 'Empresa Ejemplo', 'proveedor': None}


## 1. Preparar el conjunto de datos

Ejemplo: `proyecto_documentos/entrenamiento/Facturas/factura_01.pdf`, `.../Recibos/recibo_01.pdf`, `.../Contratos/contrato_01.pdf`. Usa documentos reales o sintéticos distintos, no copies del mismo archivo. El entrenamiento utilizará **las etiquetas de las carpetas**; las reglas anteriores se conservan solo como referencia y para extraer campos, **no participan en la predicción**.


In [5]:
from collections import Counter
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import pandas as pd
import shutil
import zipfile
import hashlib

ENTRENAMIENTO = BASE / "entrenamiento"
MODELO_RUTA = BASE / "modelo_documentos.joblib"
ENTRENAMIENTO.mkdir(parents=True, exist_ok=True)

textos, etiquetas, archivos_validos, errores = [], [], [], []
for carpeta_categoria in sorted(ENTRENAMIENTO.iterdir()):
    if not carpeta_categoria.is_dir():
        continue
    categoria = carpeta_categoria.name
    for archivo in sorted(carpeta_categoria.rglob("*")):
        if not archivo.is_file() or archivo.suffix.lower() not in EXTENSIONES:
            continue
        try:
            paginas, avisos = leer_documento(archivo)
            texto = " ".join(p.get("texto", "") for p in paginas).strip()
            if len(texto) < 30:
                errores.append(f"{archivo}: texto insuficiente; {avisos}")
                continue
            textos.append(texto)
            etiquetas.append(categoria)
            archivos_validos.append(str(archivo))
        except Exception as exc:
            errores.append(f"{archivo}: {exc}")

conteo = Counter(etiquetas)
print("Documentos útiles:", len(textos))
print("Documentos por categoría:", dict(conteo))
if errores:
    print("Archivos omitidos:")
    for error in errores[:20]: print(" -", error)
if len(conteo) < 2:
    raise ValueError("Se requieren al menos dos categorías con documentos legibles en entrenamiento/.")
if min(conteo.values()) < 5:
    raise ValueError("Para una evaluación mínima, agrega al menos 5 documentos legibles por categoría.")


Documentos útiles: 104
Documentos por categoría: {'Comprobante': 12, 'Contratos': 19, 'Facturas': 18, 'Formulario': 14, 'Ordenes_compra': 19, 'Recibos': 22}


## 2. Entrenar y evaluar

Separamos documentos en entrenamiento (80 %) y prueba (20 %). El vectorizador se ajusta **solamente con entrenamiento** porque está dentro del `Pipeline`: esto evita la fuga de información. La precisión que obtengas depende de tus propios datos y no se garantiza un porcentaje particular.


In [6]:
X_ent, X_prueba, y_ent, y_prueba = train_test_split(
    textos, etiquetas, test_size=0.20, random_state=42, stratify=etiquetas
)
modelo = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5),
        sublinear_tf=True, max_features=20000
    )),
    ("clasificador", LogisticRegression(
        max_iter=1500, class_weight="balanced", random_state=42
    ))
])
modelo.fit(X_ent, y_ent)
predicciones = modelo.predict(X_prueba)
print(classification_report(y_prueba, predicciones, zero_division=0))
print("Matriz de confusión:")
clases = sorted(set(etiquetas))
display(pd.DataFrame(confusion_matrix(y_prueba, predicciones, labels=clases),
                     index=clases, columns=clases))
joblib.dump(modelo, MODELO_RUTA)
print("Modelo entrenado guardado en:", MODELO_RUTA.resolve())


                precision    recall  f1-score   support

   Comprobante       0.50      0.50      0.50         2
     Contratos       1.00      1.00      1.00         4
      Facturas       0.75      0.75      0.75         4
    Formulario       1.00      1.00      1.00         3
Ordenes_compra       0.75      0.75      0.75         4
       Recibos       0.50      0.50      0.50         4

      accuracy                           0.76        21
     macro avg       0.75      0.75      0.75        21
  weighted avg       0.76      0.76      0.76        21

Matriz de confusión:


,Comprobante,Contratos,Facturas,Formulario,Ordenes_compra,Recibos
Comprobante,1,0,1,0,0,0
Contratos,0,4,0,0,0,0
Facturas,0,0,3,0,0,1
Formulario,0,0,0,3,0,0
Ordenes_compra,0,0,0,0,3,1
Recibos,1,0,0,0,1,2


Modelo entrenado guardado en: C:\Users\mcgre\Desktop\organizador_notebook_web\modelo_documentos.joblib


## 3. Clasificar documentos nuevos

Coloca los archivos que deseas ordenar en `proyecto_documentos/documentos/`. El modelo entrenado se carga desde `modelo_documentos.joblib`; **no necesita las plantillas al clasificar**. Una predicción con baja seguridad se envía a `Revision_necesaria`; el umbral 0.55 es orientativo, no está calibrado como garantía de acierto.


In [7]:
modelo_cargado = joblib.load(MODELO_RUTA)  # Carga únicamente archivos .joblib generados por ti.
UMBRAL_REVISION = 0.55

def predecir_texto(texto):
    if len(texto.strip()) < 30:
        return "Revision_necesaria", 0.0
    probabilidades = modelo_cargado.predict_proba([texto])[0]
    indice = int(probabilidades.argmax())
    prob = float(probabilidades[indice])
    categoria = str(modelo_cargado.classes_[indice])
    if prob < UMBRAL_REVISION:
        categoria = "Revision_necesaria"
    return categoria, prob

rutas = sorted(p for p in ENTRADA.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONES)
registros = []
ZIP_RUTA = SALIDA / "documentos_organizados.zip"
with zipfile.ZipFile(ZIP_RUTA, 'w', compression=zipfile.ZIP_DEFLATED) as zip_salida:
    for ruta in rutas:
        try:
            paginas, avisos = leer_documento(ruta)
            texto = " ".join(p.get("texto", "") for p in paginas)
            categoria, confianza = predecir_texto(texto)
            identificador = hashlib.sha256(str(ruta.relative_to(ENTRADA)).encode()).hexdigest()[:8]
            nombre = f"{ruta.stem}_{identificador}{ruta.suffix.lower()}"
            destino = f"{categoria}/{nombre}"
            zip_salida.write(ruta, arcname=destino)
            registros.append({"archivo": str(ruta.relative_to(ENTRADA)),
                              "categoria": categoria, "confianza_orientativa": round(confianza, 3),
                              "destino": destino, "avisos": "; ".join(avisos)})
        except Exception as exc:
            registros.append({"archivo": str(ruta.relative_to(ENTRADA)),
                              "categoria": "Error", "confianza_orientativa": 0,
                              "destino": "", "avisos": str(exc)})
    tabla = pd.DataFrame(registros, columns=["archivo", "categoria", "confianza_orientativa", "destino", "avisos"])
    zip_salida.writestr("resumen_clasificacion.csv", tabla.to_csv(index=False).encode("utf-8-sig"))
display(tabla)
print("Archivo ZIP:", ZIP_RUTA.resolve())


,archivo,categoria,confianza_orientativa,destino,avisos


Archivo ZIP: C:\Users\mcgre\Desktop\organizador_notebook_web\resultados\documentos_organizados.zip


In [18]:
import importlib
import motor_notebook as motor

motor = importlib.reload(motor)

print("Idioma:", motor.IDIOMA_OCR)

archivo = next(
    Path("entrenamiento/Comprobante").glob("*.jpg")
)

resultado = motor.diagnosticar_documento(
    archivo.name,
    archivo.read_bytes()
)

print("Caracteres:", resultado["caracteres"])
print("Categoría:", resultado["categoria"])
print("Probabilidades:", resultado["probabilidades"])
print("Avisos:", resultado["avisos"])

Tesseract disponible: True
Idiomas instalados: {'spa', 'osd', 'eng'}
Idioma utilizado: spa+eng
Funciones cargadas: buscar, clasificar, estrategias_imagen, extraer_campos, leer_documento, limpiar_texto, normalizar, ocr_imagen
Modelo cargado correctamente.
Categorías: ['Comprobante' 'Contratos' 'Facturas' 'Formulario' 'Ordenes_compra'
 'Recibos']
--------------------------------------
ORGANIZADOR DE DOCUMENTOS
--------------------------------------
Notebook: C:\Users\mcgre\Desktop\organizador_notebook_web\ID_documentos_11.ipynb
Modelo: C:\Users\mcgre\Desktop\organizador_notebook_web\modelo_documentos.joblib
OCR disponible: True
Idioma OCR: spa+eng
Umbral: 0.0
Motor preparado correctamente.
--------------------------------------
Idioma: spa+eng
Caracteres: 1876
Categoría: Comprobante
Probabilidades: {'Comprobante': 0.3816, 'Contratos': 0.1165, 'Facturas': 0.1357, 'Formulario': 0.117, 'Ordenes_compra': 0.1148, 'Recibos': 0.1345}
Avisos: []


In [21]:

import importlib
import motor_notebook as motor
from pathlib import Path

motor = importlib.reload(motor)

archivo = next(
    Path("entrenamiento/Comprobante").glob("*.jpg")
)

with __import__("tempfile").TemporaryDirectory() as temp:
    ruta = Path(temp) / archivo.name
    ruta.write_bytes(archivo.read_bytes())

    paginas, avisos = motor.leer_documento(ruta)

    print("Páginas detectadas:", len(paginas))
    print("Avisos:", avisos)

    for i, pagina in enumerate(paginas):
        texto = pagina.get("texto", "")

        print(f"\nPágina {i+1}")
        print("Caracteres:", len(texto))
        print("Texto:", repr(texto[:300]))


Tesseract disponible: True
Idiomas instalados: {'spa', 'osd', 'eng'}
Idioma utilizado: spa+eng
Funciones cargadas: buscar, clasificar, estrategias_imagen, extraer_campos, leer_documento, limpiar_texto, normalizar, ocr_imagen
Modelo cargado correctamente.
Categorías: ['Comprobante' 'Contratos' 'Facturas' 'Formulario' 'Ordenes_compra'
 'Recibos']
--------------------------------------
ORGANIZADOR DE DOCUMENTOS
--------------------------------------
Notebook: C:\Users\mcgre\Desktop\organizador_notebook_web\ID_documentos_11.ipynb
Modelo: C:\Users\mcgre\Desktop\organizador_notebook_web\modelo_documentos.joblib
OCR disponible: True
Idioma OCR: spa+eng
Umbral: 0.0
Motor preparado correctamente.
--------------------------------------
Páginas detectadas: 1
Avisos: []

Página 1
Caracteres: 1876
Texto: 'BBVA\n\nNAGIBE BERENICE MORENO DURAN\nRTNO 37 DE GENARO GARCIA 25\nJDN BALBUENA\nVENUSTIANO CARRANZA\nCIUDAD DE MEXICO\n\nMEXICO CP 15900\n\nInformación Financiera\n\nRendimiento\nSaldo Promedio\n

In [22]:

from PIL import Image

imagen = Image.open(archivo).convert("RGB")

print("OCR disponible:", motor.OCR_OK)
print("Idioma:", motor.IDIOMA_OCR)

texto, confianza, metodo = motor.ocr_imagen(imagen)

print("\nMétodo:", metodo)
print("Confianza OCR:", confianza)
print("Caracteres:", len(texto))
print("Texto extraído:", repr(texto[:500]))


OCR disponible: True
Idioma: spa+eng

Método: ocr_1_psm3
Confianza OCR: 89.3
Caracteres: 1876
Texto extraído: 'BBVA\n\nNAGIBE BERENICE MORENO DURAN\nRTNO 37 DE GENARO GARCIA 25\nJDN BALBUENA\nVENUSTIANO CARRANZA\nCIUDAD DE MEXICO\n\nMEXICO CP 15900\n\nInformación Financiera\n\nRendimiento\nSaldo Promedio\n\nEstado de Cuenta\n\nLIBRETON NOMINA\nPAGINA 1/7\nPeriodo DEL 29/01/2022 AL 28/02/2022\nFecha de Corte 28/02/2022\nNo. de Cuenta 2764339851\nNo. de Cliente 00927334\nRFC MODN850525-D4A\nNo, Cuenta CLABE 012 180 02764339851 0\nSUCURSAL: 5022 ARCOS\nDIRECCION: PASEO DE LOS TAMARINDOS 400 A COL.\nBOSQUES DE LAS LOMAS MEX\n\nPLAZA: CI'


In [ ]:
import importlib
import motor_notebook as motor

motor = importlib.reload(motor)

pred, paginas, avisos = motor.procesar_documento(
    archivo.name,
    archivo.read_bytes()
)

print("CATEGORÍA:", pred["categoria"])
print("PUNTUACIÓN:", pred["coincidencia"])
print("ESTADO:", pred["estado"])
print("PÁGINAS:", paginas)
print("AVISOS:", avisos)

Tesseract disponible: True
Idiomas instalados: {'spa', 'osd', 'eng'}
Idioma utilizado: spa+eng
Funciones cargadas: buscar, clasificar, estrategias_imagen, extraer_campos, leer_documento, limpiar_texto, normalizar, ocr_imagen
Modelo cargado correctamente.
Categorías: ['Comprobante' 'Contratos' 'Facturas' 'Formulario' 'Ordenes_compra'
 'Recibos']
--------------------------------------
ORGANIZADOR DE DOCUMENTOS
--------------------------------------
Notebook: C:\Users\mcgre\Desktop\organizador_notebook_web\ID_documentos_11.ipynb
Modelo: C:\Users\mcgre\Desktop\organizador_notebook_web\modelo_documentos.joblib
OCR disponible: True
Idioma OCR: spa+eng
Umbral: 0.0
Motor preparado correctamente.
--------------------------------------
CATEGORÍA: Comprobante
PUNTUACIÓN: 38.16
ESTADO: Clasificado
PÁGINAS: 1
AVISOS: []


: 